## Imports and project setup

This section includes all required libraries used in the project.  
They support data loading, preprocessing, feature scaling, regression modeling, and evaluation.  
The project compares several linear regression-based approaches, including regularized models such as Ridge, Lasso, and ElasticNet.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Data download and loading

The dataset is downloaded directly from Kaggle and loaded into a pandas DataFrame.  
This makes the notebook reproducible and allows the analysis to be run from scratch without manually uploading files.

In [57]:
import kagglehub
import os
path = kagglehub.dataset_download("kundanbedmutha/exam-score-prediction-dataset")
print("Path to dataset files:", path)
print(os.listdir(path))

Path to dataset files: /Users/janzawadzki/.cache/kagglehub/datasets/kundanbedmutha/exam-score-prediction-dataset/versions/2
['Exam_Score_Prediction.csv']


In [58]:
file_path = os.path.join(path, 'Exam_Score_Prediction.csv')
df = pd.read_csv(file_path)

## Initial data inspection

At this stage, the dataset is explored to understand its structure, identify data types, and check for missing values.  
This step helps determine what preprocessing is needed before modeling and provides a general overview of the available features.

In [59]:
df.head()

,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [60]:
df.isna().sum()

student_id          0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [61]:
df.dtypes

student_id            int64
age                   int64
gender               object
course               object
study_hours         float64
class_attendance    float64
internet_access      object
sleep_hours         float64
sleep_quality        object
study_method         object
facility_rating      object
exam_difficulty      object
exam_score          float64
dtype: object

## Data preprocessing

In this section, categorical variables are transformed into a numerical format suitable for regression models.  
Ordinal features such as facility rating, exam difficulty, and sleep quality are mapped to ordered numeric values, while nominal variables such as course, study method, and gender are encoded using one-hot encoding.

Boolean columns are then converted into integers, and the identifier column is removed because it does not carry predictive value.

In [62]:
df['facility_rating'] = df['facility_rating'].map({'high': 2, 'medium': 1, 'low': 0})
df['exam_difficulty'] = df['exam_difficulty'].map({'hard': 2, 'moderate': 1, 'easy': 0})
df['sleep_quality'] = df['sleep_quality'].map({'good': 2, 'average': 1, 'poor': 0})
df['internet_access'] = df['internet_access'].map({'yes': 1, 'no': 0})

In [63]:
df = pd.get_dummies(df, columns=["course"], drop_first=True)
df = pd.get_dummies(df, columns=["study_method"], drop_first=True)
df = pd.get_dummies(df, columns=["gender"], drop_first=True)

In [64]:
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

In [65]:
df.drop(columns='student_id',inplace=True)

## Feature and target selection

The target variable in this project is `exam_score`, while all remaining columns are treated as predictor features.  
This step defines the supervised learning setup used for the regression task.

In [ ]:
X = df.drop(columns=["exam_score"])
y = df["exam_score"]

## Train-test split

The dataset is divided into training and test subsets.  
The training set is used to fit the models, while the test set is reserved for evaluating how well the models generalize to unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Feature scaling

Feature scaling is introduced to standardize the input variables.  
This is especially important for regularized regression models such as Ridge, Lasso, and ElasticNet, because their penalty terms are sensitive to the scale of the features.

In [83]:
scaler = StandardScaler()
scaler.fit_transform(X_train)
scaler.transform(X_test)

array([[ 0.23722635, -0.86639587, -1.27217958, ..., -0.51110944,
        -0.70677535,  1.40301821],
       [ 1.55013591,  1.10503239, -1.40552627, ..., -0.51110944,
        -0.70677535, -0.71274912],
       [-1.51331974,  0.11281904, -1.68961269, ...,  1.95652815,
        -0.70677535,  1.40301821],
       ...,
       [ 0.23722635,  0.52876874, -1.43451468, ...,  1.95652815,
        -0.70677535,  1.40301821],
       [-1.51331974,  0.06082533,  0.31058763, ..., -0.51110944,
        -0.70677535,  1.40301821],
       [ 0.23722635, -1.0180442 , -0.35614581, ..., -0.51110944,
         1.41487673, -0.71274912]], shape=(4000, 20))

## Baseline model: Linear Regression

A standard Linear Regression model is trained first to provide a baseline for comparison.  
Its results serve as a reference point for evaluating whether regularization improves predictive performance.

In [84]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", mean_squared_error(y_test, y_pred_lr) ** 0.5)
print("R2:", r2_score(y_test, y_pred_lr))

Linear Regression
MAE: 7.86205475438177
RMSE: 9.770710275800266
R2: 0.7331071422534656


## Regularized regression models

To extend the baseline approach, three regularized models are tested: Ridge, Lasso, and ElasticNet.  
Different values of `alpha` are used to examine how the strength of regularization affects prediction quality.

This makes it possible to compare whether coefficient shrinkage improves performance or whether the simpler linear model is already sufficient.

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

### Ridge Regression

Ridge Regression applies L2 regularization, which shrinks coefficient values without eliminating them completely.  
It is useful when predictors may be correlated and helps control model complexity.

In [89]:
for a in alphas:
    ridge = Ridge(alpha = a)
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)    

    print(f"Ridge - Alpha: {a}")
    print("MAE:", mean_absolute_error(y_test, y_pred_ridge))
    print("RMSE:", mean_squared_error(y_test, y_pred_ridge) ** 0.5)
    print("R2:", r2_score(y_test, y_pred_ridge))
    print("-"*30)

Ridge - Alpha: 0.01
MAE: 7.862054626795618
RMSE: 9.770709819522988
R2: 0.7331071671804443
------------------------------
Ridge - Alpha: 0.1
MAE: 7.862053479084441
RMSE: 9.770705725065959
R2: 0.7331073908655249
------------------------------
Ridge - Alpha: 1
MAE: 7.862042057732034
RMSE: 9.770665968784796
R2: 0.7331095627938011
------------------------------
Ridge - Alpha: 10
MAE: 7.861989793975182
RMSE: 9.770383799352006
R2: 0.7331249777585744
------------------------------
Ridge - Alpha: 100
MAE: 7.868142592716528
RMSE: 9.776317864008117
R2: 0.7328007050201806
------------------------------


### Lasso Regression

Lasso Regression uses L1 regularization, which can shrink some coefficients all the way to zero.  
This means it not only regularizes the model, but can also perform a simple form of feature selection.

In [91]:
for a in alphas:
    lasso = Lasso(alpha = a)
    lasso.fit(X_train, y_train)
    y_pred_lasso = lasso.predict(X_test)    

    print(f"Lasso - Alpha: {a}")
    print("MAE:", mean_absolute_error(y_test, y_pred_lasso))
    print("RMSE:", mean_squared_error(y_test, y_pred_lasso) ** 0.5)
    print("R2:", r2_score(y_test, y_pred_lasso))
    print("-"*30)

Lasso - Alpha: 0.01
MAE: 7.861771055882218
RMSE: 9.768839891386413
R2: 0.7332093138351832
------------------------------
Lasso - Alpha: 0.1
MAE: 7.900695984101392
RMSE: 9.8117695101228
R2: 0.730859313543445
------------------------------
Lasso - Alpha: 1
MAE: 8.536066085335454
RMSE: 10.559883651264848
R2: 0.6882525233014678
------------------------------
Lasso - Alpha: 10
MAE: 10.295923862924106
RMSE: 12.667422419411482
R2: 0.5513980191130354
------------------------------
Lasso - Alpha: 100
MAE: 15.501150492747612
RMSE: 18.869035561387598
R2: 0.004631055674062523
------------------------------


### ElasticNet Regression

ElasticNet combines both L1 and L2 penalties, balancing the behavior of Ridge and Lasso.  
This model is useful when a compromise between coefficient shrinkage and feature selection is desired.

In [96]:
for a in alphas:
    en = ElasticNet(alpha = a, l1_ratio=0.5)
    en.fit(X_train, y_train)
    y_pred_en = en.predict(X_test)    

    print(f"ElaticNet - Alpha: {a}")
    print("MAE:", mean_absolute_error(y_test, y_pred_en))
    print("RMSE:", mean_squared_error(y_test, y_pred_en) ** 0.5)
    print("R2:", r2_score(y_test, y_pred_en))
    print("-"*30)

ElaticNet - Alpha: 0.01
MAE: 7.866895469079764
RMSE: 9.77437572727684
R2: 0.7329068566464281
------------------------------
ElaticNet - Alpha: 0.1
MAE: 8.032442316536159
RMSE: 9.966908973056515
R2: 0.7222809541930868
------------------------------
ElaticNet - Alpha: 1
MAE: 8.78323573250702
RMSE: 10.826716489103214
R2: 0.6722986659225774
------------------------------
ElaticNet - Alpha: 10
MAE: 11.581337033794314
RMSE: 14.209763008990889
R2: 0.43550727372070175
------------------------------
ElaticNet - Alpha: 100
MAE: 15.095974587801033
RMSE: 18.336848015803714
R2: 0.059986585286390226
------------------------------


## Model evaluation

To compare the regression models, three evaluation metrics were used:

- **MAE (Mean Absolute Error)** – the average absolute difference between predicted and actual exam scores,
- **RMSE (Root Mean Squared Error)** – a metric that penalizes larger errors more strongly,
- **R² (Coefficient of Determination)** – the proportion of variance in the target variable explained by the model.

These metrics were used to evaluate the baseline **Linear Regression** model and compare it with the regularized models: **Ridge**, **Lasso**, and **ElasticNet**.

### Linear Regression

Linear Regression was used as the baseline model.

| Model | MAE   | RMSE  | R²    |
|:------|------:|------:|------:|
| Linear Regression | 7.862 | 9.771 | 0.7331 |

The baseline model explains about **73.3% of the variance** in exam scores and makes an average prediction error of about **7.9 points**.  
This is already a strong result and suggests that the relationship between the predictors and the target is largely linear.

---

### Ridge Regression

Ridge Regression was tested with multiple values of the regularization parameter `alpha`.

| Alpha | MAE   | RMSE  | R²    |
|------:|------:|------:|------:|
| 0.01  | 7.862 | 9.771 | 0.7331 |
| 0.1   | 7.862 | 9.771 | 0.7331 |
| 1     | 7.862 | 9.771 | 0.7331 |
| 10    | 7.862 | 9.770 | 0.7331 |
| 100   | 7.868 | 9.776 | 0.7328 |

The results are almost identical to standard Linear Regression for small and moderate values of `alpha`.  
This suggests that additional L2 regularization was not necessary in this problem.  
Only for `alpha = 100` did the performance begin to decline slightly.

---

### Lasso Regression

Lasso Regression was tested with the same range of `alpha` values.

| Alpha | MAE   | RMSE  | R²    |
|------:|------:|------:|------:|
| 0.01  | 7.862 | 9.769 | 0.7332 |
| 0.1   | 7.901 | 9.812 | 0.7309 |
| 1     | 8.536 | 10.560 | 0.6883 |
| 10    | 10.296 | 12.667 | 0.5514 |
| 100   | 15.501 | 18.869 | 0.0046 |

Lasso achieved its best result for **alpha = 0.01**, where it slightly outperformed the baseline model.  
However, as the regularization strength increased, the results worsened rapidly.  
This indicates that stronger L1 regularization removed too much useful information and oversimplified the model.

---

### ElasticNet Regression

ElasticNet was evaluated using `l1_ratio = 0.5` and several `alpha` values.

| Alpha | MAE   | RMSE  | R²    |
|------:|------:|------:|------:|
| 0.01  | 7.867 | 9.774 | 0.7329 |
| 0.1   | 8.032 | 9.967 | 0.7223 |
| 1     | 8.783 | 10.827 | 0.6723 |
| 10    | 11.581 | 14.210 | 0.4355 |
| 100   | 15.096 | 18.337 | 0.0600 |

ElasticNet followed a similar pattern to Lasso.  
With very weak regularization, the model performed reasonably well, but larger values of `alpha` noticeably reduced model quality.  
This suggests that the dataset does not benefit from stronger combined L1 and L2 regularization.

---

### Best-performing model

The best result was achieved by **Lasso Regression with alpha = 0.01**.

| Best Model | Alpha | MAE   | RMSE  | R²    |
|:-----------|------:|------:|------:|------:|
| Lasso Regression | 0.01 | 7.862 | 9.769 | 0.7332 |

Although the improvement over Linear Regression and Ridge is very small, it is still the strongest result among all tested models.

---

### Final interpretation

The overall comparison shows that:

- **Linear Regression** already provides a strong baseline,
- **Ridge** performs almost identically and does not introduce any meaningful improvement,
- **Lasso** performs best only with very weak regularization,
- **ElasticNet** does not outperform the simpler models in this task.

This suggests that the problem is largely linear and that the available features already provide a strong signal for predicting exam scores.  
In this case, stronger regularization generally reduces performance rather than improving it.